# SigFlow-Sim: Signature + Conditional Normalizing Flow

This notebook implements a simplified but mathematically faithful version of the architecture described:

1. **Path Signatures (Rough Path Theory)** using `iisignature`
2. **Conditional Normalizing Flow** using `nflows`
3. **Custom Loss Function** combining likelihood, distribution matching, autocorrelation, and diversity
4. **Training Loop** using Adam optimizer

Designed as a research / learning scaffold rather than production code.

In [ ]:
# Install required libraries (run once)
# !pip install numpy pandas torch matplotlib iisignature nflows yfinance

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

import iisignature

from nflows.flows import Flow
from nflows.distributions.normal import StandardNormal
from nflows.transforms.base import CompositeTransform
from nflows.transforms.autoregressive import MaskedAffineAutoregressiveTransform

## Phase 1 — Load Market Data and Compute Log Returns

In [ ]:
import yfinance as yf

ticker = "AAPL"
data = yf.download(ticker, period="2y")

prices = data["Close"]
log_returns = np.log(prices / prices.shift(1)).dropna()

log_returns.head()

## Phase 2 — Compute Path Signatures

In [ ]:
WINDOW = 20
DEPTH = 3

def compute_signature(window):
    t = np.linspace(0, 1, len(window))
    path = np.column_stack([t, window])

    sig_stream = iisignature.prepare(path.shape[1], DEPTH)
    sig = iisignature.sig(path, sig_stream)

    return sig

signatures = []
targets = []

returns_array = log_returns.values

for i in range(WINDOW, len(returns_array) - 1):
    window = returns_array[i-WINDOW:i]
    sig = compute_signature(window)

    signatures.append(sig)
    targets.append(returns_array[i])

X = torch.tensor(np.array(signatures), dtype=torch.float32)
Y = torch.tensor(np.array(targets), dtype=torch.float32).unsqueeze(1)

print("Signature shape:", X.shape)
print("Target shape:", Y.shape)

## Phase 3 — Build Conditional Normalizing Flow

In [ ]:
def build_flow(context_dim):
    transforms = []

    for _ in range(4):
        transforms.append(
            MaskedAffineAutoregressiveTransform(
                features=1,
                hidden_features=32,
                context_features=context_dim
            )
        )

    transform = CompositeTransform(transforms)
    base_dist = StandardNormal([1])

    flow = Flow(transform, base_dist)

    return flow

flow = build_flow(X.shape[1])
optimizer = optim.Adam(flow.parameters(), lr=1e-3)

## Phase 4 — Custom Loss Components

In [ ]:
def autocorrelation(x):
    if len(x) < 2:
        return torch.tensor(0.0)

    x1 = x[:-1]
    x2 = x[1:]

    return torch.cov(torch.stack([x1.squeeze(), x2.squeeze()]))[0,1] / torch.var(x)


def custom_loss(flow, x_context, y_real):
    log_prob = flow.log_prob(y_real, context=x_context)

    loss_mle = -log_prob.mean()

    y_fake = flow.sample(len(y_real), context=x_context)

    mu_real = y_real.mean()
    mu_fake = y_fake.mean()

    sigma_real = y_real.std()
    sigma_fake = y_fake.std()

    loss_dist = (mu_real - mu_fake)**2 + (sigma_real - sigma_fake)**2

    rho_real = autocorrelation(y_real)
    rho_fake = autocorrelation(y_fake)

    loss_acf = (rho_real - rho_fake)**2

    variance_fake = torch.var(y_fake)
    loss_div = 1 / (variance_fake + 1e-6)

    total_loss = (
        loss_mle
        + 0.1 * loss_dist
        + 0.1 * loss_acf
        + 0.01 * loss_div
    )

    return total_loss

## Phase 5 — Training Loop

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64

dataset_size = len(X)

for epoch in range(EPOCHS):
    permutation = torch.randperm(dataset_size)

    total_epoch_loss = 0

    for i in range(0, dataset_size, BATCH_SIZE):
        indices = permutation[i:i+BATCH_SIZE]

        batch_x = X[indices]
        batch_y = Y[indices]

        optimizer.zero_grad()

        loss = custom_loss(flow, batch_x, batch_y)

        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_epoch_loss:.4f}")

## Phase 6 — Generate Future Scenarios

In [ ]:
def generate_scenarios(flow, context, n=100):
    samples = flow.sample(n, context=context.unsqueeze(0).repeat(n, 1))
    return samples.detach().numpy()

latest_context = X[-1]

scenarios = generate_scenarios(flow, latest_context, 200)

plt.hist(scenarios, bins=40)
plt.title("Generated Return Distribution")
plt.show()